# DDOS dataset download (Drone Depth and Obstacle Segmentation)

Downloads [benediktkol/DDOS](https://huggingface.co/datasets/benediktkol/DDOS) from Hugging Face
into `datasets/DDOS/`, keeping the original folder layout, then checks that it's complete and
shows a few samples.

**What's in it:** synthetic aerial drone frames (1280×720 PNG), 300 train / 20 val / 20 test
flights of 100 frames each (30k / 2k / 2k images), split across two environments
(`neighbourhood`, `park`). Each frame has these modalities:

| Folder | Content | Size (full set) |
|---|---|---|
| `image/` | RGB frame | ~49 GB |
| `segmentation/` | 8-bit semantic mask, pixel value = class (table in section 5) | ~1.4 GB |
| `depth/` | uint16 PNG, 0..65535 = 0..100 m linear | ~20 GB |
| `flow/` | optical flow to the next frame | ~10 GB |
| `surfacenormals/` | surface normal map | ~54 GB |

Each flight also has `metadata.csv` (pose, GPS, camera parameters per frame) and `weather.csv`.

**Note on task type:** the masks are **semantic** (one class value per pixel, no instance ids).
Training YOLO segmentation needs a conversion step to polygon labels, which this notebook
doesn't do. It only downloads and checks the data.

**Pipeline:**
1. Environment checks
2. Configuration (which modalities / splits to download)
3. Download (resumable)
4. Verify: file counts per split / environment / flight, matching files across modalities
5. Class pixel statistics on the segmentation masks
6. Visual spot check

**Environment:**
- Kernel: `yolo-env` (`envs/yolo-env`). Uses `huggingface_hub` + `hf-xet` (already installed).
- License: **CC BY-NC 4.0** (non-commercial use only).
- No login needed (the dataset is public, not gated). A Hugging Face token (`HF_TOKEN` env var or
  `hf auth login`) raises the rate limit, which helps here: the default selection is ~100k
  separate files.

## 1. Environment checks

In [ ]:
import os
import shutil
import huggingface_hub
from ultralytics.utils import SETTINGS

print("huggingface_hub:", huggingface_hub.__version__)
print("HF token set:", huggingface_hub.get_token() is not None)
os.makedirs(SETTINGS["datasets_dir"], exist_ok=True)
print("datasets_dir:", SETTINGS["datasets_dir"])
print("Free disk (GB):", round(shutil.disk_usage(SETTINGS["datasets_dir"]).free / 1e9))

## 2. Configuration

| Setting | Here | Why |
|---|---|---|
| `MODALITIES` | `image`, `segmentation`, `depth` | What segmentation and depth training need (~70 GB). Add `flow` / `surfacenormals` for the full ~134 GB. |
| `SPLITS` | all three | `test` is for the final evaluation only. Don't use it for validation or early stopping. |
| `MAX_FLIGHTS` | `None` | Set to e.g. `1` for a quick smoke test (downloads flight ids `0..MAX_FLIGHTS-1` per split/environment). |
| `MAX_WORKERS` | 16 | Parallel file downloads. |

Re-running with more modalities or splits only downloads what's missing.

In [ ]:
from pathlib import Path

REPO_ID = "benediktkol/DDOS"
REVISION = "1ed1314d32ef3a5a7e1434000783a8433517bd0e"   # pinned commit (2024-04-26), so re-runs get identical data

MODALITIES = ["image", "segmentation", "depth"]   # also available: "flow", "surfacenormals"
SPLITS = ["train", "validation", "test"]
ENVIRONMENTS = ["neighbourhood", "park"]
MAX_FLIGHTS = None                                 # None = all flights; small int for a smoke test
MAX_WORKERS = 16

DDOS_DIR = Path(SETTINGS["datasets_dir"]) / "DDOS"   # raw download, original HF layout under data/

# Expected flights per split/environment (from the dataset card: 250+50 train, 15+5 val/test)
EXPECTED_FLIGHTS = {
    "train": {"neighbourhood": 250, "park": 50},
    "validation": {"neighbourhood": 15, "park": 5},
    "test": {"neighbourhood": 15, "park": 5},
}
FRAMES_PER_FLIGHT = 100

if MAX_FLIGHTS is None:
    flight_globs = ["*"]
else:
    flight_globs = [str(i) for i in range(MAX_FLIGHTS)]
ALLOW_PATTERNS = ["README.md"] + [
    f"data/{s}/*/{f}/{x}"
    for s in SPLITS
    for f in flight_globs
    for x in [*(f"{m}/*" for m in MODALITIES), "metadata.csv", "weather.csv"]
]
print("Download to:", DDOS_DIR)
print("Modalities:", MODALITIES)
print("Splits:", SPLITS, "| max flights per env:", MAX_FLIGHTS or "all")

## 3. Download

`snapshot_download` into `DDOS_DIR`. Already-downloaded files are skipped, so an interrupted run
resumes when you re-run the cell. The cell also retries by itself a few times on network errors.

- First it lists the repo (~170k files), which takes a minute or two before any progress bar
  moves.
- Anonymous downloads are rate-limited per file request. If you see HTTP 429 warnings, the
  library waits and retries. Setting `HF_TOKEN` makes this much faster.
- **Long run** (~70 GB by default). From tmux:
  `jupyter nbconvert --to notebook --execute --inplace ddos_download.ipynb`

In [ ]:
import time
from huggingface_hub import snapshot_download

for attempt in range(1, 6):
    try:
        snapshot_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            revision=REVISION,
            local_dir=DDOS_DIR,
            allow_patterns=ALLOW_PATTERNS,
            max_workers=MAX_WORKERS,
        )
        break
    except Exception as e:  # network drop, 5xx, too many 429s: wait and resume
        if attempt == 5:
            raise
        print(f"Attempt {attempt} failed ({type(e).__name__}: {e}); retrying in {60 * attempt}s")
        time.sleep(60 * attempt)
print("Downloaded to", DDOS_DIR)

## 4. Verify the download

For each split and environment: number of flights, then for every flight the frame count per
modality and whether all modalities have the same frame ids (e.g. `image/17.png` ↔
`segmentation/17.png`). Problems are printed as `FAIL` and the cell raises at the end, so a
scripted run stops here. A missing flight count is only a `FAIL` when `MAX_FLIGHTS` is `None`.

In [ ]:
problems = []
summary = []
for split in SPLITS:
    for env in ENVIRONMENTS:
        env_dir = DDOS_DIR / "data" / split / env
        flights = sorted((p for p in env_dir.iterdir() if p.is_dir()), key=lambda p: int(p.name)) if env_dir.exists() else []
        expected = EXPECTED_FLIGHTS[split][env] if MAX_FLIGHTS is None else min(MAX_FLIGHTS, EXPECTED_FLIGHTS[split][env])
        if len(flights) != expected:
            problems.append(f"{split}/{env}: {len(flights)} flights, expected {expected}")
        n_frames = 0
        for flight in flights:
            ids = {m: {p.stem for p in (flight / m).glob("*.png")} for m in MODALITIES}
            ref = ids[MODALITIES[0]]
            n_frames += len(ref)
            if len(ref) != FRAMES_PER_FLIGHT:
                problems.append(f"{split}/{env}/{flight.name}: {len(ref)} {MODALITIES[0]} frames, expected {FRAMES_PER_FLIGHT}")
            for m in MODALITIES[1:]:
                if ids[m] != ref:
                    problems.append(f"{split}/{env}/{flight.name}: {m} frame ids differ from {MODALITIES[0]} "
                                    f"(missing {len(ref - ids[m])}, extra {len(ids[m] - ref)})")
            for f in ("metadata.csv", "weather.csv"):
                if not (flight / f).exists():
                    problems.append(f"{split}/{env}/{flight.name}: no {f}")
        summary.append((split, env, len(flights), n_frames))

print(f"{'split':10s} {'env':13s} {'flights':>7} {'frames':>7}")
for split, env, n_f, n_img in summary:
    print(f"{split:10s} {env:13s} {n_f:7d} {n_img:7d}")
print(f"{'':10s} {'total':13s} {sum(s[2] for s in summary):7d} {sum(s[3] for s in summary):7d}")
size_gb = sum(p.stat().st_size for p in (DDOS_DIR / "data").rglob("*") if p.is_file()) / 1e9
print(f"\nOn disk: {size_gb:.1f} GB")

for p in problems[:50]:
    print("FAIL", p)
if len(problems) > 50:
    print(f"... and {len(problems) - 50} more")
if problems:
    raise RuntimeError(f"{len(problems)} problems in the download; re-run section 3 to fetch missing files")
print("OK: all flights complete")

### Decode check

Fully decodes `SANITY_SAMPLES` random frames per split (every downloaded modality) to catch
truncated files, and checks the expected formats: RGB image, 8-bit single-channel mask with only
known class values, 16-bit single-channel depth.

In [ ]:
import random
import cv2
import numpy as np

SANITY_SAMPLES = 50
CLASS_MAPPING = {   # pixel value in segmentation/*.png -> class (from the dataset card); 0 = background
    255: "ultra_thin",
    240: "thin_structures",
    220: "small_mesh",
    200: "large_mesh",
    180: "trees",
    160: "buildings",
    140: "vehicles",
    100: "animals",
    80: "other",
}
KNOWN_VALUES = {0, *CLASS_MAPPING}

rng = random.Random(0)
bad = []
for split in SPLITS:
    frames = sorted((DDOS_DIR / "data" / split).glob(f"*/*/{MODALITIES[0]}/*.png"))
    for f in rng.sample(frames, min(SANITY_SAMPLES, len(frames))):
        flight, stem = f.parent.parent, f.stem
        for m in MODALITIES:
            p = flight / m / f"{stem}.png"
            a = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
            if a is None:
                bad.append(f"{p}: cannot decode")
            elif m == "image" and not (a.ndim == 3 and a.dtype == np.uint8):
                bad.append(f"{p}: image shape {a.shape} {a.dtype}")
            elif m == "segmentation":
                if a.ndim != 2 or a.dtype != np.uint8:
                    bad.append(f"{p}: mask shape {a.shape} {a.dtype}")
                elif unknown := set(np.unique(a).tolist()) - KNOWN_VALUES:
                    bad.append(f"{p}: unknown mask values {sorted(unknown)}")
            elif m == "depth" and not (a.ndim == 2 and a.dtype == np.uint16):
                bad.append(f"{p}: depth shape {a.shape} {a.dtype}")
for b in bad:
    print("FAIL", b)
if bad:
    raise RuntimeError(f"{len(bad)} bad files; delete them and re-run section 3")
print(f"OK: {SANITY_SAMPLES} frames per split decoded, formats as expected")

## 5. Class statistics (segmentation masks)

Per class: share of all pixels and share of frames containing the class, on a random sample of
`STATS_SAMPLES` train frames and all val frames. Thin structures (wires, cables) cover a tiny
fraction of pixels, which matters for loss weighting and for how small polygons get after
downscaling to `imgsz`.

In [ ]:
import collections

STATS_SAMPLES = 2000   # train frames to sample (all val frames are used)

if "segmentation" not in MODALITIES:
    print("segmentation not downloaded; skipping")
else:
    def class_stats(masks):
        px, frames = collections.Counter(), collections.Counter()
        total = 0
        for p in masks:
            vals, counts = np.unique(cv2.imread(str(p), cv2.IMREAD_UNCHANGED), return_counts=True)
            px.update(dict(zip(vals.tolist(), counts.tolist())))
            frames.update(vals.tolist())
            total += counts.sum()
        return px, frames, total, len(masks)

    stats = {}
    for split, n in (("train", STATS_SAMPLES), ("validation", None)):
        masks = sorted((DDOS_DIR / "data" / split).glob("*/*/segmentation/*.png"))
        if n is not None:
            masks = random.Random(0).sample(masks, min(n, len(masks)))
        stats[split] = class_stats(masks)

    print(f"{'value':>5} {'class':16s} {'train px %':>10} {'train frames %':>14} {'val px %':>8} {'val frames %':>12}")
    for v, name in [(0, "background"), *CLASS_MAPPING.items()]:
        row = []
        for split in ("train", "validation"):
            px, frames, total, n_frames = stats[split]
            row += [100 * px[v] / max(total, 1), 100 * frames[v] / max(n_frames, 1)]
        print(f"{v:5d} {name:16s} {row[0]:10.3f} {row[1]:14.1f} {row[2]:8.3f} {row[3]:12.1f}")

## 6. Visual spot check

Random frames from train: RGB, segmentation colored per class, depth (meters, clipped at 100 m).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

N_SHOW = 4
frames = sorted((DDOS_DIR / "data" / "train").glob(f"*/*/{MODALITIES[0]}/*.png"))
picks = random.Random(1).sample(frames, min(N_SHOW, len(frames)))

values = [0, *sorted(CLASS_MAPPING)]
seg_cmap = ListedColormap(plt.cm.tab10(np.linspace(0, 1, len(values))))
seg_norm = BoundaryNorm([v - 0.5 for v in values] + [values[-1] + 0.5], len(values))

cols = [m for m in ("image", "segmentation", "depth") if m in MODALITIES]
fig, axes = plt.subplots(len(picks), len(cols), figsize=(6 * len(cols), 3.6 * len(picks)), squeeze=False)
for row, f in zip(axes, picks):
    flight = f.parent.parent
    for ax, m in zip(row, cols):
        a = cv2.imread(str(flight / m / f"{f.stem}.png"), cv2.IMREAD_UNCHANGED)
        if m == "image":
            ax.imshow(cv2.cvtColor(a, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{flight.relative_to(DDOS_DIR / 'data')} / {f.stem}")
        elif m == "segmentation":
            # Map raw values to 0..len(values)-1 so every class gets a distinct color
            ax.imshow(a, cmap=seg_cmap, norm=seg_norm, interpolation="nearest")
            ax.set_title("segmentation")
        else:
            im = ax.imshow(a.astype(np.float32) / 65535 * 100, cmap="magma_r", vmin=0, vmax=100)
            fig.colorbar(im, ax=ax, fraction=0.025, label="m")
            ax.set_title("depth")
        ax.axis("off")
if "segmentation" in cols:
    names = {0: "background", **CLASS_MAPPING}
    fig.legend(handles=[Patch(color=seg_cmap(seg_norm(v)), label=names[v]) for v in values],
               loc="lower center", ncol=len(values), frameon=False)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()